# 10주차 실습 : 토픽 모델링, 감성 분석, 그리고 기계 독해

### [1] 실습 진행 관련 설명

문서의 주제와 감성을 파악하고, 문서를 읽고 질문에 답하는 세 가지 과제를 차례로 실습한다.
실습은 크게 세 부분으로 구성되어 있다.

- **[실습 1] 토픽 모델링** : 문서 집합에서 잠재적인 주제를 찾아낸다.
- **[실습 2] 감성 분석** : 문장의 긍정/부정을 분류하는 모델을 학습한다.
- **[실습 3] 기계 독해** : 지문을 읽고 질문에 답하는 모델을 사용하고 채점한다.

각 실습은 **라이브러리를 이용한 방법**으로 먼저 결과를 얻는다.
그중 [실습 1]과 [실습 3]은 핵심 계산을 **직접 함수로 구현**하여 라이브러리 내부 동작까지 확인한다.

In [ ]:
# [0] 환경 설정  ─ 런타임을 새로 시작했다면 이 셀을 가장 먼저 실행한다.
#     torch 는 Colab 에 이미 설치되어 있으므로 다시 설치하지 않는다
#     (버전 미지정 설치는 CUDA 빌드를 CPU 빌드로 바꿔 버릴 수 있다).
!pip install -q -U "transformers==4.48.3" > /dev/null 2>&1
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False
    print('한글 폰트 설정 완료')
except Exception as e:
    print('[경고] 한글 폰트 설정 실패 :', e)


### [2] 코드 구현의 흐름

- **(1) (데이터 측면)** 데이터 불러오기 및 확인
- **(2) (표현 측면)** 텍스트를 수치 벡터로 변환하기 (BoW / TF-IDF)
- **(3) (모델 측면)** 알고리즘 불러오기 또는 직접 구현하기
- **(4) (학습 과정)** 모델 학습 또는 분해 수행하기
- **(5) (성능 평가)** 결과를 해석하고 지표로 평가하기

세 실습 모두 이 흐름을 그대로 따른다.

## (공통) 환경 설정

Colab에는 `scikit-learn`, `pandas`, `matplotlib`이 이미 설치되어 있다.
[실습 3]에서 사용하는 `transformers`만 아래 셀에서 설치한다.
로컬 주피터 환경이라면 주석 처리된 줄의 주석을 풀고 한 번만 실행하면 된다.

In [ ]:
# [실습 3] 기계 독해에서 사용하는 패키지 설치 (Colab 기준 1~2분 소요)
!pip install -q transformers torch

# 로컬 주피터 환경이라면 아래 주석도 풀고 한 번만 실행한다.
# !pip install -q scikit-learn pandas matplotlib textblob

In [ ]:
# (공통) 라이브러리 불러오기
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.max_colwidth', 80)

print('numpy  :', np.__version__)
print('pandas :', pd.__version__)

---
# [실습 1] 토픽 모델링 (Topic Modeling)

문서 집합에서 **어떤 주제들이 들어 있는지**를 비지도 학습으로 찾아낸다.

수행 순서
1. 데이터 불러오기
2. 문서-단어 행렬 만들기 (CountVectorizer / TfidfVectorizer)
3. LSA (TruncatedSVD) 수행하기
4. LDA (LatentDirichletAllocation) 수행하기
5. numpy로 LSA를 직접 구현해 결과 비교하기
6. 토픽 개수 K를 바꿔가며 비교하기

## 1-1. (데이터 측면) 데이터 불러오기

실습용 문서 집합은 **스포츠 / 금융 / 요리** 세 가지 주제의 짧은 문서 24개로 구성되어 있다.
주제 레이블은 결과를 확인하기 위한 용도일 뿐, **토픽 모델링 학습에는 사용하지 않는다** (비지도 학습).

In [ ]:
# (1) (데이터 측면) 데이터 불러오기
# 외부 파일 없이 바로 실행할 수 있도록 문서를 코드 안에 정의한다.

documents = [
    # --- 스포츠 (sports) ---
    "The football team won the championship match with a late goal",
    "Our soccer coach changed the team formation before the final game",
    "The player scored three goals and the crowd cheered in the stadium",
    "A basketball game went into overtime and the team won by two points",
    "The athlete broke the world record at the olympic stadium yesterday",
    "Fans filled the stadium to watch the final match of the season",
    "The coach said the team needs more practice before the next match",
    "He scored the winning goal in the last minute of the football game",

    # --- 금융 (finance) ---
    "The stock market fell sharply after the central bank raised interest rates",
    "Investors moved money into bonds as the market became more volatile",
    "The company reported strong quarterly earnings and the stock price rose",
    "Analysts expect the interest rate decision to affect the currency market",
    "The bank announced a new investment fund for retail investors",
    "Inflation data pushed the stock index lower during morning trading",
    "The investor sold shares of the technology company at a high price",
    "Market volatility increased as earnings season began this week",

    # --- 요리 (cooking) ---
    "Add the chopped onion and garlic to the pan and cook for five minutes",
    "Mix the flour sugar and butter until the dough becomes smooth",
    "Bake the cake in a preheated oven for thirty minutes at low heat",
    "The recipe requires fresh tomato olive oil and a little salt",
    "Boil the pasta in salted water and then add the tomato sauce",
    "Chop the vegetables finely and fry them in a hot pan with oil",
    "This dessert recipe uses sugar butter and eggs mixed together",
    "Season the soup with salt and pepper and let it cook slowly",
]

# 결과 확인용 정답 레이블 (학습에는 사용하지 않음)
true_labels = ['sports'] * 8 + ['finance'] * 8 + ['cooking'] * 8

print('전체 문서 수 :', len(documents))
print('주제 종류    :', sorted(set(true_labels)))
print()
for i in range(3):
    print(f'[{true_labels[i]:8s}] {documents[i]}')

### (참고) 실제 데이터셋으로 바꾸어 실행하기

아래 셀의 주석을 풀면 scikit-learn에 내장된 20 Newsgroups 데이터셋으로 동일한 실습을
수행할 수 있다 (인터넷 연결 필요). 문서 수가 많아지면 토픽이 훨씬 뚜렷해진다.

In [ ]:
# (선택) 20 Newsgroups 데이터셋 사용하기
# 아래 주석을 모두 풀면 실제 데이터셋으로 동일한 실습을 수행할 수 있다.
# true_labels 도 함께 채워지므로 이후 셀을 수정 없이 그대로 실행할 수 있다.

# from sklearn.datasets import fetch_20newsgroups
#
# categories = ['rec.sport.baseball', 'sci.med', 'talk.politics.mideast']
# news = fetch_20newsgroups(subset='train', categories=categories,
#                           remove=('headers', 'footers', 'quotes'))
# keep = [i for i, d in enumerate(news.data) if len(d.split()) > 20][:300]
# documents   = [news.data[i] for i in keep]
# true_labels = [news.target_names[news.target[i]] for i in keep]
# N_TOPICS_SUGGEST = len(categories)
# print('문서 수 :', len(documents), '/ 주제 수 :', N_TOPICS_SUGGEST)

## 1-2. (표현 측면) 문서-단어 행렬 만들기

텍스트는 그대로 사용할 수 없으므로 **문서-단어 행렬(Document-Term Matrix)** 로 변환한다.

- `CountVectorizer` : 단어의 등장 횟수를 그대로 사용 (BoW)
- `TfidfVectorizer` : 흔한 단어의 가중치를 낮춘 TF-IDF 값을 사용

In [ ]:
# (2) (표현 측면) BoW 행렬 만들기
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(stop_words='english',   # 영어 불용어 제거
                            min_df=1,               # 최소 1개 문서에 등장
                            max_df=0.9)             # 90% 넘는 문서에 등장하면 제외
X_count = count_vec.fit_transform(documents)

print('문서-단어 행렬 크기 (문서 수 x 어휘 수) :', X_count.shape)
print('어휘 예시 :', count_vec.get_feature_names_out()[:15])

# 앞의 5개 문서, 앞의 10개 단어만 표로 확인
pd.DataFrame(X_count.toarray()[:5, :10],
             columns=count_vec.get_feature_names_out()[:10],
             index=[f'doc{i}' for i in range(5)])

In [ ]:
# (2-2) (표현 측면) TF-IDF 행렬 만들기
tfidf_vec = TfidfVectorizer(stop_words='english', min_df=1, max_df=0.9)
X_tfidf = tfidf_vec.fit_transform(documents)

print('TF-IDF 행렬 크기 :', X_tfidf.shape)

# IDF 값이 큰(=희귀한) 단어와 작은(=흔한) 단어를 확인해보자
idf = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out())
print('\n[IDF가 작은 단어 = 여러 문서에 흔히 등장]')
print(idf.sort_values().head(5))
print('\n[IDF가 큰 단어 = 특정 문서에만 등장]')
print(idf.sort_values(ascending=False).head(5))

## 1-3. (모델 측면) LSA — 잠재 의미 분석

LSA는 문서-단어 행렬을 **특이값 분해(SVD)** 로 분해해 잠재 토픽을 찾는다.

$$A \approx U \Sigma V^T$$

- `TruncatedSVD`의 `components_` : 토픽-단어 행렬 (각 토픽을 구성하는 단어의 가중치)
- `fit_transform()`의 반환값 : 문서-토픽 행렬 (각 문서가 어떤 토픽과 가까운지)

In [ ]:
# (3) (모델 측면) LSA 불러오기 / (4) (학습 과정) 분해 수행
from sklearn.decomposition import TruncatedSVD

N_TOPICS = 3   # 토픽 개수 K (하이퍼파라미터)
# LSA의 첫 성분(성분 0)은 '전체 평균 방향'이라 토픽 구분에 쓰지 않는다.
# 따라서 토픽 K개를 실제로 쓰려면 K+1개를 뽑아야 한다.
N_TOPICS_LSA = N_TOPICS + 1

lsa = TruncatedSVD(n_components=N_TOPICS_LSA, random_state=42)
doc_topic_lsa = lsa.fit_transform(X_tfidf)      # 문서-토픽 행렬

print('문서-토픽 행렬 :', doc_topic_lsa.shape)
print('토픽-단어 행렬 :', lsa.components_.shape)
print('설명된 분산 비율 :', lsa.explained_variance_ratio_)
print('설명된 분산 합계 : {:.3f}'.format(lsa.explained_variance_ratio_.sum()))

In [ ]:
# (참고) LSA 성분의 부호 정규화
#   SVD 가 돌려주는 특이벡터의 부호는 임의이다. 부호가 뒤집히면 "성분 0 이 평균 방향"
#   같은 설명이 그대로 뒤집히므로, 각 성분의 문서 점수 합이 양수가 되도록 부호를 맞춘다.

import numpy as np

sign = np.sign(doc_topic_lsa.sum(axis=0))
sign[sign == 0] = 1
doc_topic_lsa = doc_topic_lsa * sign
lsa_components = lsa.components_ * sign[:, None]
print('부호 정규화 완료 — 성분별 문서 점수 합 :', np.round(doc_topic_lsa.sum(axis=0), 3))


In [ ]:
# (5) (성능 평가) 각 토픽의 상위 단어 확인하기
def print_top_words(components, feature_names, n_top=8, title='성분'):
    """토픽-단어 행렬에서 가중치가 큰 상위 단어를 출력한다."""
    for idx, comp in enumerate(components):
        top_idx = comp.argsort()[::-1][:n_top]        # 큰 값부터 정렬
        words = [feature_names[i] for i in top_idx]
        print(f'{title} {idx} : ' + ', '.join(words))

feature_names = tfidf_vec.get_feature_names_out()
print('=== LSA 결과 ===')
print_top_words(lsa_components, feature_names, n_top=8)

In [ ]:
# (5-2) (성능 평가) 각 문서가 어떤 토픽에 배정되었는지 확인
#
# [주의] LSA의 첫 번째 성분(토픽 0)은 '전체 문서의 평균적인 방향'을 담는 경우가 많아
#        거의 모든 문서에서 값이 가장 크게 나온다. 그대로 argmax를 하면 모든 문서가
#        토픽 0으로 몰려 결과 해석이 어렵다.
#        따라서 여기서는 첫 번째 성분을 제외하고 나머지 성분으로 배정한다.
#        (LDA는 결과가 확률이므로 이런 처리가 필요 없다 - 두 방법의 중요한 차이이다.)
assigned_raw = doc_topic_lsa.argmax(axis=1)
assigned = doc_topic_lsa[:, 1:].argmax(axis=1)   # 성분 0 제외 → 라벨 0 ~ N_TOPICS-1

print('성분 0 포함 배정 :', np.bincount(assigned_raw, minlength=N_TOPICS_LSA))
print('성분 0 제외 배정 :', np.bincount(assigned, minlength=N_TOPICS))

result_lsa = pd.DataFrame({
    '문서': [d[:45] + '...' for d in documents],
    '정답주제': true_labels,
    'LSA토픽': assigned,
})
result_lsa.head(24)

In [ ]:
# (5-3) (성능 평가) 정답 주제와 LSA 토픽의 교차표
# 비지도 학습이므로 토픽 번호와 주제 이름이 일치할 필요는 없다.
# 각 주제가 하나의 토픽에 몰려 있다면 잘 학습된 것이다.
pd.crosstab(pd.Series(true_labels, name='정답 주제'),
            pd.Series(assigned, name='LSA 토픽'))

## 1-4. (모델 측면) LDA — 잠재 디리클레 할당

LDA는 문서가 여러 토픽의 **확률적 혼합**으로 생성되었다고 가정한다.
LSA와 달리 결과가 **확률 값**이므로 해석이 직관적이다.

LDA는 단어의 등장 횟수를 사용하는 모델이므로 TF-IDF가 아니라 **CountVectorizer 행렬**을 입력한다.

In [ ]:
# (3) (모델 측면) LDA 불러오기 / (4) (학습 과정) 학습 수행
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,      # 토픽 개수 K
    doc_topic_prior=None,       # alpha (None이면 1/K)
    topic_word_prior=None,      # beta  (None이면 1/K)
    learning_method='batch',
    max_iter=50,
    random_state=42,
)
doc_topic_lda = lda.fit_transform(X_count)      # 문서-토픽 확률 행렬

print('문서-토픽 행렬 :', doc_topic_lda.shape)
print('행 합계 (확률이므로 1) :', doc_topic_lda.sum(axis=1)[:5])

In [ ]:
# (5) (성능 평가) LDA 토픽별 상위 단어
count_feature_names = count_vec.get_feature_names_out()
print('=== LDA 결과 ===')
print_top_words(lda.components_, count_feature_names, n_top=8)

In [ ]:
# (5-2) (성능 평가) 문서별 토픽 확률 분포 확인
prob = pd.DataFrame(doc_topic_lda,
                    columns=[f'토픽{i}' for i in range(N_TOPICS)])
prob['정답주제'] = true_labels
prob['배정토픽'] = doc_topic_lda.argmax(axis=1)
prob.round(3).head(12)

In [ ]:
# (5-3) (성능 평가) 문서-토픽 분포 시각화
plt.figure(figsize=(9, 6))
plt.imshow(doc_topic_lda, aspect='auto', cmap='Blues')
plt.colorbar(label='topic probability')
plt.xlabel('topic')
plt.ylabel('document index')
plt.title('LDA document-topic distribution')
plt.xticks(range(N_TOPICS))
# 주제가 바뀌는 경계선 표시
for b in [7.5, 15.5]:
    plt.axhline(b, color='red', linewidth=1)
plt.tight_layout()
plt.show()

## 1-5. (직접 구현) numpy로 LSA 재현하기

`TruncatedSVD`가 내부에서 무엇을 하는지 확인하기 위해, numpy의 `np.linalg.svd`로
직접 특이값 분해를 수행해 본다.

$$A = U \Sigma V^T \quad \rightarrow \quad
\text{상위 } K \text{개만 사용} \quad A \approx U_K \Sigma_K V_K^T$$

In [ ]:
# (3) (모델 측면) LSA 직접 구현하기

def my_lsa(X, k):
    """희소 행렬 X를 받아 상위 k개 성분으로 절단된 SVD를 수행한다.

    반환
      doc_topic  : 문서-토픽 행렬 (n_docs x k)
      topic_word : 토픽-단어 행렬 (k x n_terms)
      s          : 특이값 (토픽 중요도, 길이 k)
    """
    A = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)

    # full_matrices=False 로 두면 U:(n x r), s:(r,), Vt:(r x m) 형태가 된다.
    U, s, Vt = np.linalg.svd(A, full_matrices=False)

    # 특이값이 큰 상위 k개만 남긴다 (= 절단된 SVD)
    U_k, s_k, Vt_k = U[:, :k], s[:k], Vt[:k, :]

    doc_topic = U_k * s_k          # U_k @ diag(s_k) 와 동일
    topic_word = Vt_k
    return doc_topic, topic_word, s_k


# sklearn 의 lsa 와 같은 성분 수를 써야 특이값·상위 단어를 1:1로 비교할 수 있다
my_doc_topic, my_topic_word, sing = my_lsa(X_tfidf, N_TOPICS_LSA)

print('특이값 (토픽 중요도) :', sing.round(3))
print('문서-토픽 행렬 :', my_doc_topic.shape)
print('토픽-단어 행렬 :', my_topic_word.shape)

In [ ]:
# (5) (성능 평가) 직접 구현한 LSA의 토픽 상위 단어
print('=== 직접 구현한 LSA 결과 ===')
print_top_words(my_topic_word, feature_names, n_top=8)

print()
print('sklearn TruncatedSVD 결과와 비교')
print('- 특이벡터(U의 열, Vt의 행)의 부호는 SVD 계산 방식에 따라 통째로 뒤집힐 수 있어,')
print('  같은 토픽이라도 상위 단어가 반대쪽(음수 방향)에서 나올 수 있다. 특이값 자체는 항상 0 이상이다.')
print('sklearn 특이값 :', lsa.singular_values_.round(3))
print('직접 구현 특이값 :', sing.round(3))

In [ ]:
# (참고) 토픽 공간에서의 문서 유사도
# 원래 단어 공간(수백 차원) 대신 저차원 토픽 공간에서 코사인 유사도를 계산한다.
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(doc_topic_lsa)

query = 0   # 0번 문서(스포츠)와 가장 비슷한 문서 찾기
order = sim[query].argsort()[::-1][1:4]

print('기준 문서 :', documents[query])
print()
for rank, i in enumerate(order, 1):
    print(f'{rank}위 (유사도 {sim[query, i]:.3f}) [{true_labels[i]}] {documents[i]}')

## 1-6. 토픽 개수 K 바꿔보기

K는 분석자가 직접 정해야 하는 하이퍼파라미터이다.
K를 바꿔가며 토픽의 상위 단어가 어떻게 달라지는지 확인한다.

In [ ]:
# K = 2, 3, 4, 5 로 바꾸어 LDA 결과 비교하기
for k in [2, 3, 4, 5]:
    m = LatentDirichletAllocation(n_components=k, max_iter=30,
                                  learning_method='batch', random_state=42)
    m.fit(X_count)
    print(f'\n----- K = {k} (perplexity = {m.perplexity(X_count):.1f}) -----')
    print_top_words(m.components_, count_feature_names, n_top=6)

---
# [실습 2] 감성 분석 (Sentiment Analysis)

문장의 **긍정(1) / 부정(0)** 을 분류하는 지도 학습 모델을 만든다.

수행 순서
1. 데이터 불러오기 및 확인
2. 학습/테스트 데이터 분할
3. TF-IDF로 피처 추출
4. 로지스틱 회귀 모델 학습
5. 정확도, 정밀도, 재현율, F1로 평가
6. 새로운 문장 예측 및 모델 해석

## 2-1. (데이터 측면) 데이터 불러오기

UCI **Sentiment Labelled Sentences** 데이터셋을 사용한다.
Amazon과 Yelp가 각 1,000문장, IMDb가 748문장으로 모두 2,748문장이다.
인터넷 연결이 되지 않는 환경에서도 실습할 수 있도록, 다운로드에 실패하면
노트북에 포함된 예비 데이터로 자동 전환된다.

In [ ]:
# (1) (데이터 측면) 데이터 불러오기
COLUMN_NAMES = ['Review', 'Sentiment']
DATA_URL = ('https://raw.githubusercontent.com/'
            'PacktWorkshops/The-Natural-Language-Processing-Workshop/'
            'master/Chapter08/data/sentiment_labelled_sentences/')

def load_uci_sentiment():
    """UCI 감성 데이터셋을 불러온다. 실패하면 None을 반환한다."""
    try:
        frames = []
        for name in ['imdb_labelled.txt', 'amazon_cells_labelled.txt',
                     'yelp_labelled.txt']:
            df = pd.read_table(DATA_URL + name, names=COLUMN_NAMES)
            df['Source'] = name.split('_')[0]
            frames.append(df)
        return pd.concat(frames, ignore_index=True)
    except Exception as e:
        print('[알림] 온라인 데이터 불러오기 실패 →', type(e).__name__)
        return None


reviews = load_uci_sentiment()
print('온라인 데이터 사용' if reviews is not None else '예비 데이터 사용 예정')

In [ ]:
# (1-2) (데이터 측면) 예비(내장) 데이터셋
# 온라인 데이터를 사용할 수 없을 때를 대비한 소규모 데이터이다.
BACKUP_POS = [
    "This movie was absolutely wonderful and I enjoyed every minute",
    "The food was delicious and the service was excellent",
    "Great phone with a fantastic battery life and clear screen",
    "I love this product it works perfectly and arrived quickly",
    "The staff were friendly and the atmosphere was very pleasant",
    "An amazing performance by the whole cast highly recommended",
    "Sound quality is superb and the price is very reasonable",
    "Best purchase I have made this year truly satisfied",
    "The plot was engaging and the ending was very satisfying",
    "Comfortable fit and the material feels durable and soft",
    "Fast delivery and the item was exactly as described",
    "Beautiful design and it is very easy to use",
    "The chicken was tender and full of flavor we will return",
    "Excellent camera quality even in low light conditions",
    "A brilliant film that kept me interested from start to finish",
    "Customer support solved my problem quickly and politely",
    "The battery lasts all day which is exactly what I needed",
    "Wonderful experience and I would definitely come back again",
    "Crisp display and smooth performance for the price",
    "The dessert was heavenly and beautifully presented",
]
BACKUP_NEG = [
    "This movie was terrible and a complete waste of my time",
    "The food was cold and the service was extremely slow",
    "The battery died within an hour which is very disappointing",
    "I hate this product it broke after two days of use",
    "Rude staff and the restaurant was dirty and noisy",
    "A boring film with a predictable and pointless ending",
    "Sound quality is awful and the volume is far too low",
    "Worst purchase I have ever made do not buy this",
    "The story made no sense and the acting was poor",
    "The material feels cheap and it tore on the first day",
    "Delivery took three weeks and the item was damaged",
    "Ugly design and the buttons are impossible to press",
    "The chicken was dry and tasteless we will not return",
    "Camera quality is horrible and the photos are blurry",
    "A dull movie that I could not finish watching",
    "Customer support ignored my emails for over a week",
    "The battery drains too fast and it overheats badly",
    "Awful experience and I will never come back here",
    "The screen froze constantly and the phone kept crashing",
    "The dessert was stale and tasted like cardboard",
]

if reviews is None:
    print('⚠ 다운로드 실패 — 예비 데이터로 진행합니다.')
    print('   아래 정확도·혼동행렬은 강의자료의 2,748문장 결과와 다릅니다.')
    reviews = pd.DataFrame({
        'Review': BACKUP_POS + BACKUP_NEG,
        'Sentiment': [1] * len(BACKUP_POS) + [0] * len(BACKUP_NEG),
        'Source': 'backup',
    })

print('전체 문장 수 :', len(reviews))
reviews.head(5)

In [ ]:
# (1-3) (데이터 측면) 레이블 분포 확인
# 부정적인 리뷰는 감성 점수 0, 긍정적인 리뷰는 감성 점수 1을 갖는다.
counts = reviews.Sentiment.value_counts().to_frame()
counts.index = pd.Series(['Negative' if i == 0 else 'Positive' for i in counts.index])
print(counts)

plt.figure(figsize=(4, 3))
plt.bar(['Negative(0)', 'Positive(1)'],
        [ (reviews.Sentiment == 0).sum(), (reviews.Sentiment == 1).sum() ])
plt.title('sentiment label distribution')
plt.ylabel('count')
plt.tight_layout()
plt.show()

## 2-2. (학습 과정) 학습 / 테스트 데이터 분할

모델이 **처음 보는 문장**에 대해 잘 동작하는지 확인해야 하므로,
데이터를 학습용과 테스트용으로 나눈다. `stratify` 옵션으로 두 집합의 클래스 비율을 맞춘다.

In [ ]:
# (2) 데이터 분할
from sklearn.model_selection import train_test_split

X_text = reviews['Review'].values
y = reviews['Sentiment'].values

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y)

print('학습 데이터 :', len(X_train_txt))
print('테스트 데이터 :', len(X_test_txt))
print('학습 데이터 긍정 비율 : {:.2f}'.format(y_train.mean()))

## 2-3. (표현 측면) 피처 추출

리뷰 텍스트는 그대로 사용할 수 없으므로 **피처 벡터**로 변환한다.

주의 : 벡터화 도구는 **학습 데이터로만 `fit`** 해야 한다.
테스트 데이터까지 함께 `fit`하면 테스트 정보가 새어 들어가 성능이 부풀려진다.

In [ ]:
# (3) (표현 측면) TF-IDF 피처 추출
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,          # 'not', 'no' 등이 감성에 중요하므로 불용어를 제거하지 않는다
    ngram_range=(1, 2),       # 단어 1개 + 연속된 2개 (예: 'not good')
    min_df=1,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(X_train_txt)   # 학습 데이터로 fit + transform
X_test = vectorizer.transform(X_test_txt)         # 테스트 데이터는 transform만

print('학습 피처 행렬 :', X_train.shape)
print('테스트 피처 행렬 :', X_test.shape)
print('추출된 피처 예시 :', vectorizer.get_feature_names_out()[:10])

## 2-4. (모델 측면) 감성 모델 학습

로지스틱 회귀는 5주차에 학습한 알고리즘으로, 텍스트 분류에서 가장 널리 쓰이는
기준선(baseline) 모델이다.

In [ ]:
# (4) (모델 측면 / 학습 과정) 로지스틱 회귀 학습
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model.fit(X_train, y_train)

print('학습 완료')
print('학습 데이터 정확도 : {:.3f}'.format(model.score(X_train, y_train)))
print('테스트 데이터 정확도 : {:.3f}'.format(model.score(X_test, y_test)))

## 2-5. (성능 평가) 평가 지표 확인

- **정확도(Accuracy)** = (TP + TN) / 전체
- **정밀도(Precision)** = TP / (TP + FP)
- **재현율(Recall)** = TP / (TP + FN)
- **F1** = 2 × 정밀도 × 재현율 / (정밀도 + 재현율)

In [ ]:
# (5) (성능 평가) 지표 계산
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

y_pred = model.predict(X_test)

print('정확도  : {:.3f}'.format(accuracy_score(y_test, y_pred)))
print('정밀도  : {:.3f}'.format(precision_score(y_test, y_pred)))
print('재현율  : {:.3f}'.format(recall_score(y_test, y_pred)))
print('F1 점수 : {:.3f}'.format(f1_score(y_test, y_pred)))
print()
print(classification_report(y_test, y_pred,
                            target_names=['Negative(0)', 'Positive(1)']))

In [ ]:
# (5-2) (성능 평가) 혼동 행렬 시각화
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred 0', 'pred 1'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['true 0', 'true 1'])
ax.set_title('confusion matrix')
plt.tight_layout()
plt.show()

print('TN =', cm[0, 0], '/ FP =', cm[0, 1], '/ FN =', cm[1, 0], '/ TP =', cm[1, 1])

## 2-6. (성능 평가) 새로운 문장 예측 및 모델 해석

In [ ]:
# (5-3) 처음 보는 문장의 감성 예측
new_sentences = [
    "I really love this product it works great",
    "This is the worst service I have ever received",
    "The movie was not good at all",          # 부정 표현 - 어려운 사례
    "It is okay but nothing special",         # 중립에 가까운 문장
]

new_X = vectorizer.transform(new_sentences)
pred = model.predict(new_X)
prob = model.predict_proba(new_X)[:, 1]      # 긍정일 확률

for s, p, pr in zip(new_sentences, pred, prob):
    label = '긍정' if p == 1 else '부정'
    print(f'[{label}] (긍정 확률 {pr:.3f})  {s}')

In [ ]:
# (5-4) 모델이 학습한 감성 단어 확인
# 로지스틱 회귀의 계수가 클수록 긍정, 작을수록(음수) 부정에 기여한다.
coef = pd.Series(model.coef_[0], index=vectorizer.get_feature_names_out())

top_pos = coef.sort_values(ascending=False).head(15)
top_neg = coef.sort_values().head(15)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].barh(top_pos.index[::-1], top_pos.values[::-1])
axes[0].set_title('top positive features')
axes[1].barh(top_neg.index[::-1], top_neg.values[::-1], color='indianred')
axes[1].set_title('top negative features')
plt.tight_layout()
plt.show()

In [ ]:
# (5-5) 틀린 사례 분석 - 왜 틀렸는지 살펴보자
wrong = np.where(y_pred != y_test)[0]
print(f'오분류 문장 수 : {len(wrong)} / {len(y_test)}')
print()
for i in wrong[:5]:
    print(f'정답={y_test[i]}  예측={y_pred[i]}  |  {X_test_txt[i]}')

### (선택) TextBlob 규칙 기반 감성 분석과 비교

TextBlob은 학습 없이 사전(lexicon)만으로 감성을 판정한다.
학습 기반 모델과 결과가 어떻게 다른지 비교해 보자.

In [ ]:
# (선택) TextBlob 사용 - 설치되어 있지 않으면 건너뛴다
try:
    from textblob import TextBlob

    for s in new_sentences:
        blob = TextBlob(s)
        print(f'polarity={blob.sentiment.polarity:+.3f}  '
              f'subjectivity={blob.sentiment.subjectivity:.3f}  |  {s}')
except ImportError:
    print('TextBlob이 설치되어 있지 않다. 아래 명령으로 설치할 수 있다.')
    print('!pip install -q textblob')

---
# [실습 3] 기계 독해 (Machine Reading Comprehension)

**지문(context)** 을 읽고 **질문(question)** 에 답하는 과제를 수행한다.
SQuAD로 사전학습된 모델을 HuggingFace `pipeline`으로 불러와 사용하고,
EM과 F1 지표를 직접 구현해 채점한다.

수행 순서
1. SQuAD 형식 데이터 준비하기
2. 질의응답 파이프라인 불러오기
3. 지문과 질문을 전달해 답 얻기
4. EM / F1 채점 함수 직접 구현하기
5. 전체 데이터에 대해 채점하기
6. 답할 수 없는 질문 실험하기

## 3-1. (데이터 측면) SQuAD 형식 데이터 준비

SQuAD 데이터는 `context`, `question`, `answers` 세 가지 요소로 구성되며,
원본 JSON의 `answers`에는 정답 문자열과 함께 지문에서의 시작 문자 위치인
`answer_start`가 들어 있다.

본 실습에서는 정답을 문자열 리스트로만 두고, `answer_start`가 필요한 경우
`context.find(answer)`로 직접 계산한다 (아래 셀 마지막 부분 참고).

In [ ]:
# (1) (데이터 측면) SQuAD 형식 데이터 정의
CONTEXT_1 = (
    "The Amazon rainforest is a moist broadleaf forest that covers most of the "
    "Amazon basin of South America. This basin encompasses seven million square "
    "kilometers, of which five and a half million square kilometers are covered "
    "by the rainforest. The majority of the forest is contained within Brazil, "
    "with 60 percent of the rainforest, followed by Peru with 13 percent and "
    "Colombia with 10 percent. The rainforest is home to about 390 billion "
    "individual trees divided into 16000 species."
)

CONTEXT_2 = (
    "BERT stands for Bidirectional Encoder Representations from Transformers. "
    "It was introduced by researchers at Google in 2018. BERT is pre-trained on "
    "a large corpus using two objectives, masked language modeling and next "
    "sentence prediction. After pre-training, BERT can be fine-tuned with just "
    "one additional output layer to create models for a wide range of tasks, "
    "such as question answering and language inference. On the SQuAD 1.1 "
    "benchmark, BERT achieved an F1 score of 93.2."
)

squad_examples = [
    {"context": CONTEXT_1,
     "question": "Which country contains the majority of the Amazon rainforest?",
     "answers": ["Brazil"]},
    {"context": CONTEXT_1,
     "question": "How many square kilometers does the Amazon basin encompass?",
     "answers": ["seven million square kilometers", "seven million"]},
    {"context": CONTEXT_1,
     "question": "How many tree species are in the rainforest?",
     "answers": ["16000", "16000 species"]},
    {"context": CONTEXT_2,
     "question": "What does BERT stand for?",
     "answers": ["Bidirectional Encoder Representations from Transformers"]},
    {"context": CONTEXT_2,
     "question": "Who introduced BERT?",
     "answers": ["researchers at Google", "Google"]},
    {"context": CONTEXT_2,
     "question": "What F1 score did BERT achieve on SQuAD 1.1?",
     "answers": ["93.2"]},
]

print('예제 수 :', len(squad_examples))
print()
ex = squad_examples[0]
print('[context]', ex['context'][:120], '...')
print('[question]', ex['question'])
print('[answers ]', ex['answers'])

# answer_start 확인 - 정답이 지문 안에 실제로 존재하는지 검증한다
start = ex['context'].find(ex['answers'][0])
print(f"\nanswer_start = {start}  ->  "
      f"'{ex['context'][start:start + len(ex['answers'][0])]}'")

## 3-2. (모델 측면) 질의응답 파이프라인 불러오기

`distilbert-base-cased-distilled-squad`는 SQuAD 1.1로 미세 조정된 경량 모델이다.
처음 실행할 때 모델을 내려받으므로 1~2분 정도 걸릴 수 있다.

In [ ]:
# (3) (모델 측면) HuggingFace 질의응답 파이프라인 생성
from transformers import pipeline

qa = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
)
print('파이프라인 준비 완료')

In [ ]:
# (4) (학습 과정 대신 추론) 지문과 질문을 전달해 실행
ex = squad_examples[0]
result = qa(question=ex['question'], context=ex['context'])

print('question :', ex['question'])
print('answer   :', result['answer'])
print('score    : {:.4f}'.format(result['score']))
print('start/end:', result['start'], '/', result['end'])
print()
print('지문에서 잘라낸 부분 :',
      repr(ex['context'][result['start']:result['end']]))

## 3-3. (성능 평가) EM과 F1 직접 구현하기

SQuAD 공식 평가 방식을 그대로 구현한다.

1. **정규화** : 소문자로 바꾸고, 구두점 · 관사(a, an, the) · 여분의 공백을 제거한다.
2. **EM** : 정규화한 문자열이 완전히 같으면 1, 아니면 0.
3. **F1** : 토큰 집합의 겹침 정도로 계산한다.
4. 정답이 여러 개면 **각각에 대해 계산한 값 중 최대값**을 사용한다.

In [ ]:
# (5) (성능 평가) 채점 함수 직접 구현
import string
from collections import Counter


def normalize_answer(s):
    """SQuAD 공식 평가와 동일한 정규화를 수행한다."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    return white_space_fix(remove_articles(remove_punc(s.lower())))


def exact_match_score(prediction, ground_truth):
    """완전 일치하면 1.0, 아니면 0.0"""
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score_qa(prediction, ground_truth):
    """토큰 단위 F1 점수를 계산한다."""
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        # 둘 다 비어 있으면 일치로 본다
        return float(pred_tokens == gt_tokens)

    # 겹치는 토큰 수 (중복 등장 횟수까지 고려)
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)   # 예측 중 맞은 비율
    recall = num_same / len(gt_tokens)        # 정답 중 찾아낸 비율
    return 2 * precision * recall / (precision + recall)


def metric_max_over_ground_truths(fn, prediction, ground_truths):
    """정답이 여러 개일 때 최대값을 취한다."""
    return max(fn(prediction, gt) for gt in ground_truths)


# 동작 확인
print(normalize_answer('The  Brazil, '))                      # -> 'brazil'
print(exact_match_score('Brazil', 'the Brazil'))              # -> 1.0
print(round(f1_score_qa('seven million square kilometers',
                        'seven million'), 3))                 # 부분 일치

In [ ]:
# (5-2) (성능 평가) 전체 예제 채점하기
rows = []
for ex in squad_examples:
    out = qa(question=ex['question'], context=ex['context'])
    pred = out['answer']
    em = metric_max_over_ground_truths(exact_match_score, pred, ex['answers'])
    f1 = metric_max_over_ground_truths(f1_score_qa, pred, ex['answers'])
    rows.append({
        'question': ex['question'],
        'prediction': pred,
        'gold': ex['answers'][0],
        'score': round(out['score'], 3),
        'EM': em,
        'F1': round(f1, 3),
    })

scored = pd.DataFrame(rows)
print('전체 EM : {:.3f}'.format(scored.EM.mean()))
print('전체 F1 : {:.3f}'.format(scored.F1.mean()))
scored

## 3-4. 한국어 기계 독해

동일한 방식으로 KorQuAD로 학습된 모델을 사용하면 한국어 지문에도 적용할 수 있다.
`model` 인자만 한국어 모델로 바꾸면 된다.

In [ ]:
# (선택) 한국어 질의응답
KO_CONTEXT = (
    "세종대왕은 조선의 제4대 국왕으로 1418년에 즉위하였다. "
    "세종은 1443년에 훈민정음을 창제하였고 1446년에 이를 반포하였다. "
    "훈민정음은 백성이 쉽게 배우고 쓸 수 있도록 만든 문자로, "
    "오늘날 한글의 바탕이 되었다. 세종대왕은 또한 측우기와 자격루 등 "
    "여러 과학 기구의 제작을 후원하였다."
)
KO_QUESTIONS = [
    ("훈민정음은 언제 창제되었는가?", ["1443년"]),
    ("세종대왕은 조선의 몇 대 국왕인가?", ["제4대", "4대"]),
    ("훈민정음이 반포된 해는?", ["1446년"]),
]

try:
    qa_ko = pipeline("question-answering",
                     model="monologg/koelectra-small-v2-distilled-korquad-384")
    for q, golds in KO_QUESTIONS:
        out = qa_ko(question=q, context=KO_CONTEXT)
        f1 = metric_max_over_ground_truths(f1_score_qa, out['answer'], golds)
        print(f"Q: {q}\n   A: {out['answer']}  (score={out['score']:.3f}, F1={f1:.2f})")
except Exception as e:
    print('한국어 모델을 불러오지 못했다 :', type(e).__name__)
    print('인터넷 연결이 필요하며, 다른 KorQuAD 모델로 교체해도 된다.')

## 3-5. 기계 독해의 한계 확인하기

지문에 **답이 없는 질문**을 던지면 어떻게 되는지 확인한다.
SQuAD 1.1로 학습된 모델은 '답이 없다'를 배우지 않았으므로,
반드시 지문 어딘가를 답으로 내놓는다. 이것이 SQuAD 2.0이 등장한 이유이다.

In [ ]:
# 답이 없는 질문 실험
# 앞의 두 개는 지문에 답이 없는 질문, 마지막 하나는 답이 있는 대조군이다.
probe_questions = [
    "What is the population of Brazil?",          # (답 없음) 지문에 없는 정보
    "Who is the president of Colombia?",          # (답 없음) 지문에 없는 정보
    "How many square kilometers are covered by the rainforest?",  # (답 있음) 대조군
]

for q in probe_questions:
    out = qa(question=q, context=CONTEXT_1)
    print(f"Q: {q}")
    print(f"   A: {out['answer']}   (confidence = {out['score']:.4f})")
    print()

print('→ 신뢰도(score)가 낮은 답은 신뢰할 수 없다고 판단하는 임계값 방식을 흔히 사용한다.')

---
# 과제

아래 세 과제를 수행하고, 코드와 함께 결과에 대한 **설명을 서술형으로** 작성하여 제출한다.

---

## 과제 1. 토픽 개수 K의 선택 (토픽 모델링)

1. [실습 1]의 문서 집합에 대해 K = 2, 3, 4, 5, 6으로 LDA를 학습한다.
2. 각 K에 대한 **혼란도(perplexity)** 를 계산하고 그래프로 그린다.
3. 각 K에서 토픽별 상위 단어 6개를 출력한다.
4. 정량 지표(혼란도)와 정성 평가(상위 단어가 사람이 보기에 하나의 주제로 묶이는가)를 함께 고려하여
   **적절한 K를 선택하고 그 근거를 3문장 이상 서술**한다.

> 힌트 : `lda_k = LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=30, random_state=42).fit(X_count)` 로 학습한 뒤 `lda_k.perplexity(X_count)` 를 사용한다.
> (`model` 은 [실습 2]의 로지스틱 회귀 객체이므로 변수 이름이 겹치지 않게 할 것)

---

## 과제 2. 감성 분류 성능 개선 (감성 분석)

1. [실습 2]의 파이프라인에서 아래 조건을 바꿔가며 최소 **4가지 조합**을 실험한다.
   - 벡터화 방식 : `CountVectorizer` vs `TfidfVectorizer`
   - N-gram 범위 : `(1, 1)` vs `(1, 2)`
   - 불용어 제거 : `stop_words=None` vs `stop_words='english'`
2. 각 조합의 **정확도와 F1 점수**를 표로 정리한다.
3. 가장 성능이 좋은 조합과 가장 나쁜 조합을 고르고, **왜 그런 차이가 났는지 설명**한다.
4. (심화) `LogisticRegression` 대신 `MultinomialNB` 또는 `LinearSVC`를 사용해 결과를 비교한다.

---

## 과제 3. 기계 독해 실패 사례 분석 (기계 독해)

1. 관심 있는 주제로 **5문장 이상의 지문**을 직접 작성한다 (영어 또는 한국어).
2. 그 지문에 대해 **질문 5개**와 각각의 정답을 직접 만든다.
   - 이 중 최소 1개는 **지문에 답이 없는 질문**으로 만든다.
3. 모델의 예측을 얻고, 실습에서 구현한 `exact_match_score`와 `f1_score_qa`로 채점한다.
4. 틀린 사례를 최소 2개 골라, **왜 틀렸는지 유형을 분류하고 서술**한다.
   - 예 : 여러 문장에 걸친 추론 필요 / 질문의 표현이 지문과 다름 / 답이 없는데 억지로 답함 등
5. 답이 없는 질문을 걸러내기 위한 **신뢰도 임계값**을 정하고, 그 값을 정한 근거를 설명한다.

---

### 제출물

- 실행 결과가 모두 포함된 `.ipynb` 파일 1개
- 각 과제의 서술형 답변은 마크다운 셀로 작성

---
# [과제] 10주차

아래 세 과제를 수행한 뒤 **실행 결과가 포함된 노트북**을 제출한다.

### 과제 1. 토픽 개수 K 선택 (30점)
K 를 여러 값으로 바꾸어 LSA 의 설명분산과 LDA 의 로그가능도를 비교하고,
어떤 K 를 고를지와 그 근거를 서술한다.

### 과제 2. 감성 분류 성능 개선 (30점)
벡터화 방식(Count / TF-IDF)과 `ngram_range` 를 바꾸어 4가지 이상 비교하고,
가장 좋은 조합과 그 이유를 서술한다.

### 과제 3. 직접 만든 질의응답 채점 (40점)
서로 다른 주제의 지문·질문·정답 5쌍을 직접 만들어 EM / F1 을 계산하고,
실패 사례 하나를 골라 원인을 분석한다.


In [ ]:
# [과제 1] 토픽 개수 K 를 바꾸어 비교하고 근거를 제시하시오. (30점)

def compare_topic_counts(k_list=(2, 3, 4, 6, 8)):
    """각 K 에 대해 (K, LSA 설명분산 합계, LDA 로그가능도) 를 돌려주시오."""
    raise NotImplementedError(
        '과제 1 : K 를 바꾸며 TruncatedSVD 의 explained_variance_ratio_ 합과 '
        'LatentDirichletAllocation 의 score 를 측정하시오.')


# ---- 아래 블록은 위 TODO 를 채운 뒤 정상 동작합니다 ----
try:
    import pandas as pd
    rows = compare_topic_counts()
    df = pd.DataFrame(rows, columns=['K', 'LSA 설명분산', 'LDA 로그가능도'])
    display(df)
    assert len(df) >= 3, 'K 를 3개 이상 비교해야 합니다.'
    assert df['LSA 설명분산'].nunique() > 1, '모든 K 에서 같은 값이 나왔습니다 — 계산을 확인하세요.'
    print('설명분산이 더 이상 크게 늘지 않는 지점, 또는 LDA 로그가능도가 최대인 지점을')
    print('K 선택의 근거로 삼고, 왜 그 K 를 골랐는지 마크다운에 서술하시오.')
except NotImplementedError as e:
    print('▶ 미구현 :', e)


In [ ]:
# [과제 2] N-gram 범위와 벡터화 방식을 바꾸어 감성 분류 성능을 개선하시오. (30점)

def sweep_vectorizers():
    """(벡터라이저 이름, ngram_range, 테스트 정확도) 조합을 4가지 이상 돌려주시오."""
    raise NotImplementedError(
        '과제 2 : CountVectorizer / TfidfVectorizer 와 ngram_range 를 바꾸어 '
        '로지스틱 회귀 정확도를 비교하시오.')


# ---- 아래 블록은 위 TODO 를 채운 뒤 정상 동작합니다 ----
try:
    import pandas as pd
    rows = sweep_vectorizers()
    df = pd.DataFrame(rows, columns=['벡터라이저', 'ngram_range', '정확도'])
    display(df.sort_values('정확도', ascending=False))
    assert len(df) >= 4, '조합을 4가지 이상 비교해야 합니다.'
    assert df['정확도'].nunique() > 1, '모든 조합의 정확도가 같습니다 — 설정이 반영되었는지 확인하세요.'
    best = df.loc[df['정확도'].idxmax()]
    print(f"가장 좋은 조합 : {best['벡터라이저']} {best['ngram_range']} → {best['정확도']:.4f}")
except NotImplementedError as e:
    print('▶ 미구현 :', e)


In [ ]:
# [과제 3] 직접 만든 지문·질문 5개로 EM / F1 을 채점하시오. (40점)

def my_qa_set():
    """(지문, 질문, 정답) 5개를 돌려주시오. 지문은 서로 다른 주제로 작성할 것."""
    raise NotImplementedError('과제 3 : 자신이 만든 (지문, 질문, 정답) 5개를 반환하시오.')


# ---- 아래 블록은 위 TODO 를 채운 뒤 정상 동작합니다 ----
try:
    data = my_qa_set()
    assert len(data) >= 5, '5개 이상이어야 합니다.'
    assert len({c for c, _, _ in data}) >= 3, '지문이 서로 달라야 합니다.'
    em_sum = f1_sum = 0.0
    for ctx, q, gold in data:
        pred = qa(question=q, context=ctx)['answer']
        em = exact_match_score(pred, gold)
        f1 = f1_score_qa(pred, gold)
        em_sum += em; f1_sum += f1
        print(f'질문 : {q}\n  예측 : {pred}\n  정답 : {gold}\n  EM {em:.0f}  F1 {f1:.3f}\n')
    n = len(data)
    print(f'평균 EM {em_sum / n * 100:.2f}   평균 F1 {f1_sum / n * 100:.2f}')
    print('틀린 사례를 하나 골라 왜 실패했는지(지문에 없음 / 표현 불일치 등) 서술하시오.')
except NotImplementedError as e:
    print('▶ 미구현 :', e)
